# Category 4 - Embedding Model Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares embedding models for the memory recall layer in Ally Vision v2. The goal is to justify the chosen embedding model on retrieval quality, multilingual fit, dimension flexibility, price, and platform integration — not just on a single benchmark row.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
source_urls = {
  "Qwen embedding docs": "https://docs.qwencloud.com/developer-guides/embeddings/text-embedding",
  "Qwen pricing": "https://docs.qwencloud.com/developer-guides/getting-started/pricing",
  "OpenAI embeddings guide": "https://developers.openai.com/api/docs/guides/embeddings",
  "text-embedding-ada-002": "https://developers.openai.com/api/docs/models/text-embedding-ada-002",
  "BAAI bge-m3": "https://huggingface.co/BAAI/bge-m3",
  "nomic embed text v1.5": "https://huggingface.co/nomic-ai/nomic-embed-text-v1.5",
  "Ally memory config": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py"
}

# Hardcoded public-source values only. No runtime web calls are performed in this notebook.


In [ ]:
comparison_rows = [
  {
    "Metric": "MTEB retrieval score",
    "text-embedding-v4": "68.36 @ 1024 dims [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-v3": "63.39 @ 1024 dims [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-ada-002": "61.0 [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
    "text-embedding-3-small": "62.3 [source: https://developers.openai.com/api/docs/guides/embeddings]",
    "bge-m3": "N/A aggregate; multilingual retrieval benchmarks cited on HF card [source: https://huggingface.co/BAAI/bge-m3]",
    "nomic-embed-text-v1.5": "62.28 @ 768 dims [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]"
  },
  {
    "Metric": "Dimensions",
    "text-embedding-v4": "2048,1536,1024,768,512,256,128,64 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-v3": "1024,768,512 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-ada-002": "1536 fixed [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
    "text-embedding-3-small": "N/A in cached source extract [source: https://developers.openai.com/api/docs/guides/embeddings]",
    "bge-m3": "1024 [source: https://huggingface.co/BAAI/bge-m3]",
    "nomic-embed-text-v1.5": "768 [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]"
  },
  {
    "Metric": "Language support",
    "text-embedding-v4": "100+ languages [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-v3": "50+ languages [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-ada-002": "N/A exact count [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
    "text-embedding-3-small": "Multilingual improvement noted [source: https://developers.openai.com/api/docs/guides/embeddings]",
    "bge-m3": "100+ languages [source: https://huggingface.co/BAAI/bge-m3]",
    "nomic-embed-text-v1.5": "Public card available; exact count not highlighted [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]"
  },
  {
    "Metric": "Max sequence length",
    "text-embedding-v4": "8192 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-v3": "8192 [source: https://docs.qwencloud.com/developer-guides/embeddings/text-embedding]",
    "text-embedding-ada-002": "N/A in cached source [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
    "text-embedding-3-small": "N/A in cached source [source: https://developers.openai.com/api/docs/guides/embeddings]",
    "bge-m3": "8192 [source: https://huggingface.co/BAAI/bge-m3]",
    "nomic-embed-text-v1.5": "8192 [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]"
  },
  {
    "Metric": "Price per 1M tokens",
    "text-embedding-v4": "$0.07 [source: https://docs.qwencloud.com/developer-guides/getting-started/pricing]",
    "text-embedding-v3": "$0.07 (cached historical value) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/docs/Category_3_—_Intent_Classification_LLM.ipynb]",
    "text-embedding-ada-002": "$0.10 [source: https://developers.openai.com/api/docs/models/text-embedding-ada-002]",
    "text-embedding-3-small": "N/A in cached source [source: https://developers.openai.com/api/docs/guides/embeddings]",
    "bge-m3": "Local/self-hosted; API cost N/A [source: https://huggingface.co/BAAI/bge-m3]",
    "nomic-embed-text-v1.5": "Local/API hybrid; exact API price not captured [source: https://huggingface.co/nomic-ai/nomic-embed-text-v1.5]"
  },
  {
    "Metric": "Ally Vision fit",
    "text-embedding-v4": "Current repo choice at 1024 dims [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py]",
    "text-embedding-v3": "Older plan-era reference",
    "text-embedding-ada-002": "Higher cost, extra vendor",
    "text-embedding-3-small": "Separate vendor + auth",
    "bge-m3": "Extra deployment work",
    "nomic-embed-text-v1.5": "Good open alternative, not DashScope-native"
  }
]

df = pd.DataFrame(comparison_rows)
display(df)


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels=["text-embedding-v4", "text-embedding-v3", "ada-002", "nomic-v1.5"]
values=[68.36, 63.39, 61.0, 62.28]
colors=["#4fc3f7", "#555555", "#555555", "#555555"]
fig, ax = plt.subplots(figsize=(10,5))
setup_ax(ax, 'Ally Vision v2 — Embedding MTEB Comparison')
ax.bar(labels, values, color=colors)
ax.set_ylabel('MTEB / public score', color='white')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'category4_embedding_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
categories=["Languages (proxy count)", "Price per 1M tokens USD"]
series={"text-embedding-v4": {"values": [100, 0.07], "color": "#4fc3f7"}, "text-embedding-v3": {"values": [50, 0.07], "color": "#555555"}, "ada-002": {"values": [0, 0.1], "color": "#555555"}, "bge-m3": {"values": [100, 0.0], "color": "#555555"}}
x=np.arange(len(categories))
width=0.8/max(1,len(series))
fig, ax = plt.subplots(figsize=(12,5))
setup_ax(ax, 'Ally Vision v2 — Language Support and Price Comparison')
for idx, (label, spec) in enumerate(series.items()):
    offset=(idx-(len(series)-1)/2)*width
    ax.bar(x+offset, spec['values'], width=width, label=label, color=spec['color'])
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'category4_embedding_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source | URL | Accessed via |
|---|--------|-----|-------------|
| 1 | Qwen embedding docs | https://docs.qwencloud.com/developer-guides/embeddings/text-embedding | repo-cached public URL |
| 2 | Qwen pricing | https://docs.qwencloud.com/developer-guides/getting-started/pricing | Tavily extract |
| 3 | OpenAI embeddings guide | https://developers.openai.com/api/docs/guides/embeddings | repo-cached public URL |
| 4 | text-embedding-ada-002 | https://developers.openai.com/api/docs/models/text-embedding-ada-002 | repo-cached public URL |
| 5 | BAAI bge-m3 | https://huggingface.co/BAAI/bge-m3 | repo-cached public URL |
| 6 | nomic embed text v1.5 | https://huggingface.co/nomic-ai/nomic-embed-text-v1.5 | repo-cached public URL |
| 7 | Ally memory config | https://github.com/omshivarjun27/Blind-Assistance/blob/main/shared/config/settings.py | project code URL |


## CONCLUSION

text-embedding-v4 remains the best fit for Ally Vision v2 because it combines the strongest cached MTEB value in this comparison with native DashScope integration, broad language coverage, and flexible dimensions. The project’s current 1024-dimension configuration is a practical balance between retrieval quality and SQLite storage cost.

→ Chosen for Ally Vision v2 ✅
